In [3]:
import random
random.seed(42)


AA_ONE = {"AUG" : "M",
           "UUU" : "F", "UUC" : "F",
           "UGG" : "W",
           "ACC" : "T", "ACU" : "T","ACA" : "T", "ACG": "T",
           "UUA" : "L", "UUG" : "L",
           "UAA" : "*", "UAG" : "*", "UGA" : "*"}

def gc_content(seq):
    return (seq.count("G") + seq.count("C")) / len(seq)

def find_orfs(rna, codon_map):
    assert set(rna).issubset({"A", "U", "G", "C"}), "Invalid RNA"

    orfs = []

    for frame in range(3):
        i = frame
        while i <= len(rna) - 3:
            if rna[i:i+3] == "AUG":
                start = i
                protein = ""
                j = i
                terminated = False

           

                while j <= len(rna) - 3:
                    codon = rna[j:j+3]
                    aa = codon_map.get(codon)
                    
                    if aa == "*":
                        terminated = True
                        break
                        
                    if aa:
                        protein += aa
                        
                    j += 3
                    

                    
                orf_rna = rna[start:j+3] if terminated else rna[start:]
                nt_length = len(orf_rna)
                aa_length = len(protein)
                
                score = (aa_length* 2) + (gc_content(orf_rna) * 10) 
                if not terminated:
                    score -= 10  ##explicit penalty for no-stop ORF
                orfs.append({
                            "frame": frame + 1,
                            "start": start,
                            "stop": j + 2 if terminated else None,
                            "terminated": terminated,
                            "protein": protein,
                            "nt_length": nt_length,
                            "aa_length" : aa_length,
                            "gc_content": gc_content(orf_rna),
                            "score": score
                        })
                        

                    

                i = j + 3
            else:
                i += 3

    return orfs


def point_mutation(rna):
    pos = random.randint(0, len(rna) -1)
    nts = ["A", "U", "G", "C"]
    nts.remove(rna[pos])
    mutated = rna[:pos] + random.choice(nts) + rna[pos+1:]
    return mutated, "point", pos

def deletion_mutation(rna):
    pos = random.randint(0, len(rna) -1)
    mutated = rna[:pos] + rna[pos+1:]
    return mutated, "deletion", pos

def insertion_mutation(rna):
    pos = random.randint(0, len(rna))
    nt = random.choice(["A", "U", "C", "G"])
    mutated = rna[:pos] + nt + rna[pos:]
    return mutated, "insertion", pos


def compare_orfs(original, mutated):
    lost_orfs = max(0, len(original) - len(mutated))
    
    if original and mutated:
        longest_orig = max(original, key=lambda x: len(x["protein"]))["protein"]
        longest_mut = max(mutated, key=lambda x: len(x["protein"]))["protein"]
        protein_loss = max(0, len(longest_orig) - len(longest_mut))
        
    else:
        protein_loss = 0
        
        
    return lost_orfs, protein_loss    



def severity_score(lost_orfs, protein_loss, frameshift):
    return (lost_orfs * 5) + (protein_loss * 2) + (frameshift * 10)


def run_experiment(rna, n_runs, mutation_fn):
    results = []
    original_orfs = find_orfs(rna, AA_ONE)
    
    for _ in range(n_runs):
        mutated_rna, mtype, _ = mutation_fn(rna)
        mutated_orfs = find_orfs(mutated_rna, AA_ONE)
        
        lost, loss = compare_orfs(original_orfs, mutated_orfs)
        frameshift = 1 if mtype in ("deletion", "insertion") else 0
        
        score = severity_score(lost, loss, frameshift)
        results.append(score)
        
    return results


def summarize(scores):
    return { 
            "mean" : sum(scores) / len(scores),
            "max" :max(scores),
            "min" : min(scores),
            "nonzero_rate" : sum(1 for s in scores if s > 0) / len(scores)}


rna = "AUGGGGGGG"
orfs = find_orfs(rna, AA_ONE)

pt_scores = run_experiment(rna, 10000, point_mutation)
del_scores = run_experiment(rna, 10000, deletion_mutation)
ins_scores = run_experiment(rna, 10000, insertion_mutation)

pt_summary = summarize(pt_scores)
del_summary = summarize(del_scores)
ins_summary = summarize(ins_scores)


assert del_summary["mean"] > pt_summary["mean"]
assert ins_summary["mean"] > pt_summary["mean"]

assert len(orfs) == 1
assert orfs[0]["terminated"] is False
assert orfs[0]["aa_length"]>0